# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arhxmz/Flyrank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb
import getpass

con = duckdb.connect()
hf_token = getpass.getpass("Paste your HF_TOKEN: ")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest
    FROM {REL}
""").df()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,row_count,earliest,latest
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"DESCRIBE SELECT * FROM {REL}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---------- QUERY 1: grain check ----------
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain check (should be EMPTY):")
print(grain_check)

# ---------- QUERY 2: slice row count + date span ----------
slice_summary = con.sql(f"""
    SELECT COUNT(*) AS slice_row_count,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest
    FROM {REL}
    WHERE gsc_data_available IS TRUE
""").df()
print("\nSlice (gsc_data_available IS TRUE) count + span:")
print(slice_summary)

# ---------- QUERY 3: availability ----------
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {REL}
""").df()
print("\nAvailability (IS TRUE filter):")
print(availability)

# ---------- FIVE FEATURES ----------
# Decision point: March 15. Feature window = first half of the month (known BEFORE the
# decision). Label window = second half (what happens AFTER — used only for the label).
features = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,          -- knowable: logged search ranking before the decision date
        SUM(gsc_clicks) AS total_clicks,                 -- knowable: already-occurred clicks before the decision date
        SUM(ga4_sessions) AS total_sessions,              -- knowable: already-occurred sessions before the decision date
        SUM(ga4_engaged_sessions) AS total_engaged,       -- knowable: already-recorded engagement before the decision date
        SUM(scroll_events) AS total_scroll_events         -- knowable: already-recorded behavior before the decision date
    FROM {REL}
    WHERE gsc_data_available IS TRUE AND report_date < DATE '2026-03-16'
    GROUP BY content_hash_id
""").df()
print("\nFive-feature frame (first half of March):")
print(features.head())

# ---------- THE TRAP ----------
# Build the label from the SECOND half of March (the future window relative to the features).
label_window = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS clicks_second_half
    FROM {REL}
    WHERE gsc_data_available IS TRUE AND report_date >= DATE '2026-03-16'
    GROUP BY content_hash_id
""").df()

merged = features.merge(label_window, on="content_hash_id", how="inner")
merged = merged.dropna()   # drop rows missing a feature (e.g. no logged position data)
merged["is_declining"] = (merged["clicks_second_half"] < merged["total_clicks"] * 0.8).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_X = merged[["avg_position", "total_clicks", "total_sessions", "total_engaged", "total_scroll_events"]]
y = merged["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(honest_X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"\nHonest AUC (5 real features): {honest_auc:.3f}")

# Now deliberately leak: add the SAME column the label was built from, as if it were a feature.
leaky_X = merged[["avg_position", "total_clicks", "total_sessions", "total_engaged", "total_scroll_events", "clicks_second_half"]]
X_train, X_test, y_train, y_test = train_test_split(leaky_X, y, test_size=0.3, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(X_test)[:, 1])
print(f"Leaky AUC (with clicks_second_half added): {leaky_auc:.3f}  <- jumps toward 1.0, this is the trap")

# Delete the leaked column, keep only the honest number.
print(f"\nFinal honest score kept: {honest_auc:.3f}")

Grain check (should be EMPTY):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Slice (gsc_data_available IS TRUE) count + span:
   slice_row_count   earliest     latest
0          3611061 2026-03-01 2026-03-31

Availability (IS TRUE filter):
   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0

Five-feature frame (first half of March):
            content_hash_id  avg_position  total_clicks  total_sessions  \
0  content_30fc0ffeed8d67e6      8.537617           9.0             NaN   
1  content_aba5eabbcecaa682      9.194444           0.0             NaN   
2  content_40e6b52b6e3a4fb0      9.668152           0.0             NaN   
3  content_3146e465905be54c     12.457593           0.0             NaN   
4  content_7fe2b9a962f11a67     60.788874           0.0             NaN   

   total_engaged  total_scroll_events  
0            NaN                  NaN  
1            NaN                  NaN 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Rows before dropna(): {len(merged) + merged.isna().any(axis=1).sum()}")
print(f"Rows after dropna(): {len(merged)}")

Rows before dropna(): 75222
Rows after dropna(): 75222


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.